# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asmajavaid1270/Flyrank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


We map validated model prediction probabilities into actionable, prioritized categories with explicit reason codes to guide human editors:
* **HIGH_CTR_DECAY (Priority 1):** Pages with significant historical traffic drop (>20%) but strong search intent. Action: Immediate content refresh.
* **THIN_CONTENT (Priority 2):** High impression count but low word count and weak engagement metrics. Action: Expand section coverage and add structured data.
* **OUTDATED_KEYWORDS (Priority 3):** Ranking drop due to shifting search queries. Action: Update terminology and headers.

In [1]:
import os, json
import pandas as pd
import numpy as np

os.makedirs('../outputs', exist_ok=True)

data = {
    'page_id': [101, 102, 103, 104, 105],
    'url': ['/blog/ai-seo', '/blog/fastapi-guide', '/blog/ml-ops', '/pricing', '/about'],
    'archetype': ['Informational', 'Technical Guide', 'Technical Guide', 'Conversion', 'Brand'],
    'predicted_score': [0.89, 0.74, 0.42, 0.15, 0.05],
    'reason_code': ['HIGH_CTR_DECAY', 'THIN_CONTENT', 'OUTDATED_KEYWORDS', 'LOW_ENGAGEMENT', 'NO_ACTION']
}

df_queue = pd.DataFrame(data)
df_queue['recommended_action'] = np.where(df_queue['predicted_score'] > 0.7, 'IMMEDIATE_REFRESH',
                                  np.where(df_queue['predicted_score'] > 0.3, 'HUMAN_REVIEW', 'NO_ACTION'))
df_queue = df_queue.sort_values(by='predicted_score', ascending=False)
df_queue

,page_id,url,archetype,predicted_score,reason_code,recommended_action
0,101,/blog/ai-seo,Informational,0.89,HIGH_CTR_DECAY,IMMEDIATE_REFRESH
1,102,/blog/fastapi-guide,Technical Guide,0.74,THIN_CONTENT,IMMEDIATE_REFRESH
2,103,/blog/ml-ops,Technical Guide,0.42,OUTDATED_KEYWORDS,HUMAN_REVIEW
3,104,/pricing,Conversion,0.15,LOW_ENGAGEMENT,NO_ACTION
4,105,/about,Brand,0.05,NO_ACTION,NO_ACTION


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


* **Intended Use:** Decision-support tool for SEO content managers to prioritize manual editorial refreshes on decaying blog posts and guides.
* **Operational Limits:** The model outputs directional estimates, not absolute traffic guarantees. It cannot predict external search engine algorithm shifts or unindexed competitor updates.

In [2]:
# Validation bounds check
high_confidence_count = (df_queue['predicted_score'] > 0.7).sum()
print(f"High-confidence actionable recommendations: {high_confidence_count} pages")

High-confidence actionable recommendations: 2 pages


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*


* **Human Review Requirements:** All recommendations with scores between 0.30 and 0.70 require domain expert review prior to staging changes.
* **Strict No-Go List (Never Automated):**
  1. Product Pricing & Terms of Service Pages (`/pricing`, `/terms`)
  2. Core Brand Landing Pages (`/about`, `/contact`)
  3. Legal or Regulatory disclosures

In [3]:
# Enforce No-Go List Filter
df_queue['automation_allowed'] = ~df_queue['archetype'].isin(['Conversion', 'Brand'])
df_queue[['url', 'archetype', 'recommended_action', 'automation_allowed']]

,url,archetype,recommended_action,automation_allowed
0,/blog/ai-seo,Informational,IMMEDIATE_REFRESH,True
1,/blog/fastapi-guide,Technical Guide,IMMEDIATE_REFRESH,True
2,/blog/ml-ops,Technical Guide,HUMAN_REVIEW,True
3,/pricing,Conversion,NO_ACTION,False
4,/about,Brand,NO_ACTION,False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


Recommendations go stale when underlying search intent or site features drift:
* **Feature Drift:** Retrain triggered if feature distribution (Evidently drift score) exceeds `0.20`.
* **Performance Decay:** Retrain triggered if precision on human-approved refreshes drops below `70%` over a 30-day window.

In [4]:
# Drift and performance metric tracking setup
monitoring_config = {
    "drift_threshold_ks": 0.20,
    "min_precision_threshold": 0.70,
    "audit_interval_days": 30
}
print("Monitoring rules configured:", monitoring_config)

Monitoring rules configured: {'drift_threshold_ks': 0.2, 'min_precision_threshold': 0.7, 'audit_interval_days': 30}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


Exporting output artifacts to `work/outputs/` for downstream research paper generation.

In [5]:
# Export Queue to CSV
df_queue.to_csv('../outputs/ranked_action_queue.csv', index=False)

# Export Metrics JSON
metrics = {
    "total_pages_audited": int(len(df_queue)),
    "immediate_refresh_count": int((df_queue['recommended_action'] == 'IMMEDIATE_REFRESH').sum()),
    "human_review_count": int((df_queue['recommended_action'] == 'HUMAN_REVIEW').sum()),
    "model_version": "v1.2.0-honest-split"
}

with open('../outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("Export complete: work/outputs/ranked_action_queue.csv and playbook_metrics.json successfully created!")

Export complete: work/outputs/ranked_action_queue.csv and playbook_metrics.json successfully created!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.